# .RW5 to CSV Converter

Set `RW5_FILE` in the code cell to the raw `.rw5` survey file under `Datasets/Sunday Data/Original survey files`.

Leave `BEACH = "auto"` when the filename identifies Bass or Bicentennial. Set `BEACH` manually only if the filename is ambiguous. The converted CSV is written to the beach-specific Sunday Data folder.

In [ ]:
import os, re, csv
from pathlib import Path
from datetime import datetime


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Codes").is_dir() and (candidate / "Datasets").is_dir() and (candidate / "Models").is_dir():
            return candidate
    raise FileNotFoundError("Could not find project root containing Codes, Datasets, and Models")


PROJECT_ROOT = find_project_root()
DATASETS = PROJECT_ROOT / "Datasets"

# Manual input: set RW5_FILE to the raw survey file to convert.
# Place raw surveys under Datasets/Sunday Data/Original survey files.
# Leave BEACH = "auto" when the filename contains BASS, BICENP, or BICENTP.
# Set BEACH manually to "bass" or "bicentennial" only if the filename is ambiguous.
# Converted CSVs are written to the beach-specific Sunday Data folder.
# OUTPUT_NAME_SUFFIX keeps converted survey files visually distinct from raw exports.
RW5_FILE = DATASETS / "Sunday Data" / "Original survey files" / "BASS_12-12-2025.rw5"
BEACH = "auto"
OUTPUT_NAME_SUFFIX = "(TDS)"

OUTPUT_DIR_BY_BEACH = {
    "bass": DATASETS / "Sunday Data" / "Bass",
    "bicentennial": DATASETS / "Sunday Data" / "BicantP",
}


def infer_beach_from_filename(path):
    name = Path(path).name.upper()
    if "BASS" in name:
        return "bass"
    if "BICEN" in name or "BICENT" in name:
        return "bicentennial"
    raise ValueError(
        "Could not infer beach from RW5 filename. Set BEACH to 'bass' or 'bicentennial'."
    )


selected_beach = infer_beach_from_filename(RW5_FILE) if BEACH == "auto" else BEACH.lower()
if selected_beach not in OUTPUT_DIR_BY_BEACH:
    raise ValueError(f"Unknown BEACH {BEACH!r}. Choose 'auto', 'bass', or 'bicentennial'.")

OUTPUT_DIR = OUTPUT_DIR_BY_BEACH[selected_beach]

if not RW5_FILE.exists():
    raise FileNotFoundError(f"RW5 file does not exist: {RW5_FILE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

filepath = str(RW5_FILE)
fname = RW5_FILE.name
out_stem = RW5_FILE.stem
if OUTPUT_NAME_SUFFIX and not out_stem.endswith(OUTPUT_NAME_SUFFIX):
    out_stem = f"{out_stem}{OUTPUT_NAME_SUFFIX}"
out_path = OUTPUT_DIR / f"{out_stem}.csv"
out_name = str(out_path)

print("Project root:", PROJECT_ROOT)
print("Selected beach:", selected_beach)
print("Input RW5:", RW5_FILE)
print("Output CSV:", out_path)
if out_path.exists():
    print("Warning: output CSV already exists and will be overwritten.")

FIELDNAMES = [
    # ── Identity ──────────────────────────────────────────────────────────────
    "source_file",
    "job_name",
    "point_id",

    # ── Local instrument timestamp (--DT / --TM lines) ────────────────────────
    "datetime_local",       # Python datetime object → YYYY-MM-DD HH:MM:SS

    # ── GPS UTC timestamp (G0 line) ───────────────────────────────────────────
    "datetime_utc",         # Python datetime object → YYYY-MM-DD HH:MM:SS

    # ── Raw GPS coordinates (WGS84 ellipsoidal) ───────────────────────────────
    "latitude_dd",
    "longitude_dd",
    "ellipsoidal_elev_m",

    # ── Projected coordinates (NAD83 Florida East, m) ───────────────────────
    "northing_m",
    "easting_m",
    "elevation_m",

    # ── Northing statistics ───────────────────────────────────────────────────
    "nor_min", "nor_max", "nor_avg", "nor_sd",

    # ── Easting statistics ────────────────────────────────────────────────────
    "eas_min", "eas_max", "eas_avg", "eas_sd",

    # ── Elevation statistics ──────────────────────────────────────────────────
    "elv_min", "elv_max", "elv_avg", "elv_sd",

    # ── NRMS statistics ───────────────────────────────────────────────────────
    "nrms_avg", "nrms_sd", "nrms_min", "nrms_max",

    # ── ERMS statistics ───────────────────────────────────────────────────────
    "erms_avg", "erms_sd", "erms_min", "erms_max",

    # ── HRMS statistics ───────────────────────────────────────────────────────
    "hrms_avg", "hrms_sd", "hrms_min", "hrms_max",

    # ── VRMS statistics ───────────────────────────────────────────────────────
    "vrms_avg", "vrms_sd", "vrms_min", "vrms_max",

    # ── DOP statistics ────────────────────────────────────────────────────────
    "hdop_avg", "hdop_min", "hdop_max",
    "vdop_avg", "vdop_min", "vdop_max",
    "pdop_avg", "pdop_min", "pdop_max",

    # ── AGE statistics ────────────────────────────────────────────────────────
    "age_avg", "age_min", "age_max",

    # ── Satellite count ───────────────────────────────────────────────────────
    "sats_avg", "sats_min", "sats_max",

    # ── Fix-quality summary (--HRMS line) ─────────────────────────────────────
    "fix_hrms", "fix_vrms", "fix_status", "fix_sats", "fix_age",
    "fix_pdop", "fix_hdop", "fix_vdop",  "fix_tdop", "fix_gdop",
    "fix_nrms", "fix_erms",

    # ── Valid readings ────────────────────────────────────────────────────────
    "valid_readings_xy", "valid_readings_z",

    # ── Baseline vector (G1) ──────────────────────────────────────────────────
    "g1_base_point", "g1_dx", "g1_dy", "g1_dz",

    # ── Variance / covariance (G2, G3) ───────────────────────────────────────
    "g2_vx", "g2_vy", "g2_vz",
    "g3_xy", "g3_xz", "g3_yz",

    # ── File header metadata ──────────────────────────────────────────────────
    "base_lat", "base_lon", "base_elev",
    "rover_hr_m",
    "projection",
    "equipment",
    "geoid_file",
    "rtk_method",
]


def flt(s):
    try: return float(s)
    except: return None

def sint(s):
    try: return int(float(s))
    except: return None


def parse_header(lines):
    meta = {k: None for k in ["job_name", "projection", "equipment",
                                "geoid_file", "rtk_method",
                                "base_lat", "base_lon", "base_elev",
                                "rover_hr_m"]}
    for line in lines:
        l = line.strip()
        m = re.match(r"JB,NM([^,]+)", l)
        if m: meta["job_name"] = m.group(1)

        m = re.match(r"--Projection:\s*(.+)", l)
        if m: meta["projection"] = m.group(1).strip()

        m = re.match(r"--Equipment:\s*(.+)", l)
        if m: meta["equipment"] = m.group(1).strip()

        m = re.match(r"--Geoid Separation File:\s*(.+)", l)
        if m: meta["geoid_file"] = m.group(1).strip()

        m = re.match(r"--RTK Method:\s*(.+)", l)
        if m: meta["rtk_method"] = m.group(1).strip()

        m = re.match(r"BP,PN\S+,LA([\d.\-]+),LN([\d.\-]+),EL([\d.\-]+)", l)
        if m:
            meta["base_lat"]  = flt(m.group(1))
            meta["base_lon"]  = flt(m.group(2))
            meta["base_elev"] = flt(m.group(3))

        m = re.match(r"LS,HR([\d.]+)", l)
        if m: meta["rover_hr_m"] = flt(m.group(1))

    return meta


with open(filepath, "r", errors="replace") as f:
    lines = f.readlines()

header  = parse_header(lines)
records = []

i = 0
while i < len(lines):
    line = lines[i].strip()

    gps = re.match(
        r"--GPS,PN(\S+),LA([\d.\-]+),LN([\d.\-]+),EL([\d.\-]+)", line)
    if not gps:
        i += 1
        continue

    point_id         = gps.group(1)
    latitude_dd      = flt(gps.group(2))
    longitude_dd     = flt(gps.group(3))
    ellipsoidal_elev = flt(gps.group(4))

    northing = easting = elevation = None
    if i + 1 < len(lines):
        gs = re.match(
            r"--GS,PN\S+,N\s*([\d.\-]+),E\s*([\d.\-]+),EL([\d.\-]+)",
            lines[i + 1].strip())
        if gs:
            northing  = flt(gs.group(1))
            easting   = flt(gs.group(2))
            elevation = flt(gs.group(3))

    # Initialise all fields
    dt_local = dt_utc = None
    survey_date = survey_time = gps_ts_raw = None
    g1_base = g1_dx = g1_dy = g1_dz = None
    g2_vx = g2_vy = g2_vz = None
    g3_xy = g3_xz = g3_yz = None
    nor_min = nor_max = nor_avg = nor_sd = None
    eas_min = eas_max = eas_avg = eas_sd = None
    elv_min = elv_max = elv_avg = elv_sd = None
    nrms_avg = nrms_sd = nrms_min = nrms_max = None
    erms_avg = erms_sd = erms_min = erms_max = None
    hrms_avg = hrms_sd = hrms_min = hrms_max = None
    vrms_avg = vrms_sd = vrms_min = vrms_max = None
    hdop_avg = hdop_min = hdop_max = None
    vdop_avg = vdop_min = vdop_max = None
    pdop_avg = pdop_min = pdop_max = None
    age_avg  = age_min  = age_max  = None
    sats_avg = sats_min = sats_max = None
    valid_xy = valid_z  = None
    fix_hrms = fix_vrms = fix_status = fix_sats = fix_age = None
    fix_pdop = fix_hdop = fix_vdop = fix_tdop = fix_gdop = None
    fix_nrms = fix_erms = None

    j = i + 2
    while j < len(lines):
        l = lines[j].strip()
        if re.match(r"--GPS,PN", l): break

        # G0 — UTC timestamp from GPS receiver
        m = re.match(r"G0,(\d{4}/\d{2}/\d{2} \d{2}:\d{2}:\d{2})", l)
        if m: gps_ts_raw = m.group(1)

        # G1 — baseline vector
        m = re.match(r"G1,BP(\S+),PN\S+,DX([\d.\-]+),DY([\d.\-]+),DZ([\d.\-]+)", l)
        if m: g1_base, g1_dx, g1_dy, g1_dz = m.group(1), flt(m.group(2)), flt(m.group(3)), flt(m.group(4))

        # G2 — variances
        m = re.match(r"G2,VX([\d.\-]+),VY([\d.\-]+),VZ([\d.\-]+)", l)
        if m: g2_vx, g2_vy, g2_vz = flt(m.group(1)), flt(m.group(2)), flt(m.group(3))

        # G3 — covariances
        m = re.match(r"G3,XY([\d.\-]+),XZ([\d.\-]+),YZ([\d.\-]+)", l)
        if m: g3_xy, g3_xz, g3_yz = flt(m.group(1)), flt(m.group(2)), flt(m.group(3))

        # Valid readings
        m = re.match(r"--Valid Readings: XY:\s*(\d+)\s+Z:\s*(\d+)", l)
        if m: valid_xy, valid_z = sint(m.group(1)), sint(m.group(2))

        # Northing
        m = re.match(r"--Nor Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: nor_min, nor_max = flt(m.group(1)), flt(m.group(2))
        m = re.match(r"--Nor Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)", l)
        if m: nor_avg, nor_sd = flt(m.group(1)), flt(m.group(2))

        # Easting
        m = re.match(r"--Eas Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: eas_min, eas_max = flt(m.group(1)), flt(m.group(2))
        m = re.match(r"--Eas Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)", l)
        if m: eas_avg, eas_sd = flt(m.group(1)), flt(m.group(2))

        # Elevation
        m = re.match(r"--Elv Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: elv_min, elv_max = flt(m.group(1)), flt(m.group(2))
        m = re.match(r"--Elv Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)", l)
        if m: elv_avg, elv_sd = flt(m.group(1)), flt(m.group(2))

        # NRMS
        m = re.match(r"--NRMS Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: nrms_avg, nrms_sd, nrms_min, nrms_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3)), flt(m.group(4))

        # ERMS
        m = re.match(r"--ERMS Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: erms_avg, erms_sd, erms_min, erms_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3)), flt(m.group(4))

        # HRMS
        m = re.match(r"--HRMS Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: hrms_avg, hrms_sd, hrms_min, hrms_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3)), flt(m.group(4))

        # VRMS
        m = re.match(r"--VRMS Avg:\s*([\d.\-]+)\s+SD:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: vrms_avg, vrms_sd, vrms_min, vrms_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3)), flt(m.group(4))

        # HDOP
        m = re.match(r"--HDOP Avg:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: hdop_avg, hdop_min, hdop_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3))

        # VDOP
        m = re.match(r"--VDOP Avg:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: vdop_avg, vdop_min, vdop_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3))

        # PDOP
        m = re.match(r"--PDOP Avg:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: pdop_avg, pdop_min, pdop_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3))

        # AGE
        m = re.match(r"--AGE Avg:\s*([\d.\-]+)\s+Min:\s*([\d.\-]+)\s+Max:\s*([\d.\-]+)", l)
        if m: age_avg, age_min, age_max = flt(m.group(1)), flt(m.group(2)), flt(m.group(3))

        # Satellites
        m = re.match(r"--Number of Satellites Avg:\s*(\d+)\s+Min:\s*(\d+)\s+Max:\s*(\d+)", l)
        if m: sats_avg, sats_min, sats_max = sint(m.group(1)), sint(m.group(2)), sint(m.group(3))

        # Fix-quality summary line
        m = re.match(
            r"--HRMS:([\d.]+),\s*VRMS:([\d.]+),\s*STATUS:(\w+),\s*SATS:(\d+),"
            r"\s*AGE:([\d.]+),\s*PDOP:([\d.]+),\s*HDOP:([\d.]+),\s*VDOP:([\d.]+),"
            r"\s*TDOP:([\d.]+),\s*GDOP:([\d.]+),\s*NRMS:([\d.]+),\s*ERMS:([\d.]+)", l)
        if m:
            fix_hrms, fix_vrms  = flt(m.group(1)),  flt(m.group(2))
            fix_status          = m.group(3)
            fix_sats            = sint(m.group(4))
            fix_age             = flt(m.group(5))
            fix_pdop, fix_hdop  = flt(m.group(6)),  flt(m.group(7))
            fix_vdop, fix_tdop  = flt(m.group(8)),  flt(m.group(9))
            fix_gdop            = flt(m.group(10))
            fix_nrms, fix_erms  = flt(m.group(11)), flt(m.group(12))

        # Local date / time
        m = re.match(r"--DT(\d{2}-\d{2}-\d{4})", l)
        if m: survey_date = m.group(1)
        m = re.match(r"--TM(\d{2}:\d{2}:\d{2})", l)
        if m: survey_time = m.group(1)

        j += 1

    # ── Build datetime objects ────────────────────────────────────────────────

    # Local time from --DT / --TM lines (format MM-DD-YYYY HH:MM:SS)
    if survey_date and survey_time:
        try:
            dt_local = datetime.strptime(
                f"{survey_date} {survey_time}", "%m-%d-%Y %H:%M:%S")
        except ValueError:
            dt_local = None

    # UTC time from G0 line (format YYYY/MM/DD HH:MM:SS)
    if gps_ts_raw:
        try:
            dt_utc = datetime.strptime(gps_ts_raw, "%Y/%m/%d %H:%M:%S")
        except ValueError:
            dt_utc = None

    records.append({
        "source_file":        fname,
        "job_name":           header["job_name"],
        "point_id":           point_id,
        # Stored as datetime objects, written to CSV as YYYY-MM-DD HH:MM:SS
        "datetime_local":     dt_local.strftime("%Y-%m-%d %H:%M:%S") if dt_local else None,
        "datetime_utc":       dt_utc.strftime("%Y-%m-%d %H:%M:%S")   if dt_utc   else None,
        "latitude_dd":        latitude_dd,
        "longitude_dd":       longitude_dd,
        "ellipsoidal_elev_m": ellipsoidal_elev,
        "northing_m":        northing,
        "easting_m":         easting,
        "elevation_m":        elevation,
        "nor_min": nor_min, "nor_max": nor_max, "nor_avg": nor_avg, "nor_sd": nor_sd,
        "eas_min": eas_min, "eas_max": eas_max, "eas_avg": eas_avg, "eas_sd": eas_sd,
        "elv_min": elv_min, "elv_max": elv_max, "elv_avg": elv_avg, "elv_sd": elv_sd,
        "nrms_avg": nrms_avg, "nrms_sd": nrms_sd, "nrms_min": nrms_min, "nrms_max": nrms_max,
        "erms_avg": erms_avg, "erms_sd": erms_sd, "erms_min": erms_min, "erms_max": erms_max,
        "hrms_avg": hrms_avg, "hrms_sd": hrms_sd, "hrms_min": hrms_min, "hrms_max": hrms_max,
        "vrms_avg": vrms_avg, "vrms_sd": vrms_sd, "vrms_min": vrms_min, "vrms_max": vrms_max,
        "hdop_avg": hdop_avg, "hdop_min": hdop_min, "hdop_max": hdop_max,
        "vdop_avg": vdop_avg, "vdop_min": vdop_min, "vdop_max": vdop_max,
        "pdop_avg": pdop_avg, "pdop_min": pdop_min, "pdop_max": pdop_max,
        "age_avg":  age_avg,  "age_min":  age_min,  "age_max":  age_max,
        "sats_avg": sats_avg, "sats_min": sats_min, "sats_max": sats_max,
        "fix_hrms": fix_hrms, "fix_vrms": fix_vrms, "fix_status": fix_status,
        "fix_sats": fix_sats, "fix_age":  fix_age,  "fix_pdop":  fix_pdop,
        "fix_hdop": fix_hdop, "fix_vdop": fix_vdop, "fix_tdop":  fix_tdop,
        "fix_gdop": fix_gdop, "fix_nrms": fix_nrms, "fix_erms":  fix_erms,
        "valid_readings_xy": valid_xy, "valid_readings_z": valid_z,
        "g1_base_point": g1_base, "g1_dx": g1_dx, "g1_dy": g1_dy, "g1_dz": g1_dz,
        "g2_vx": g2_vx, "g2_vy": g2_vy, "g2_vz": g2_vz,
        "g3_xy": g3_xy, "g3_xz": g3_xz, "g3_yz": g3_yz,
        "base_lat":   header["base_lat"],   "base_lon":   header["base_lon"],
        "base_elev":  header["base_elev"],  "rover_hr_m": header["rover_hr_m"],
        "projection": header["projection"], "equipment":  header["equipment"],
        "geoid_file": header["geoid_file"], "rtk_method": header["rtk_method"],
    })
    i = j

# ── Save CSV ──────────────────────────────────────────────────────────────────
with open(out_name, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(records)

print(f"Parsed {len(records)} points  →  {out_name}  ({len(FIELDNAMES)} columns)")